# 4.0 — Pipeline Metrics: PPE Compliance Accuracy & People Counting MAE

Evaluate the two **pipeline-level metrics** defined in `README.md` that complement
detection-level mAP/Precision/Recall:

1. **PPE Compliance Accuracy** (target ≥ 85%) — per-person safe/violation classification
2. **People Counting Accuracy — MAE** (target ≤ 2 persons) — average error in person count per image

Uses the best model from notebook 3.1 (`resplit-oversample-conservative`) on the **test set**.

## Setup

In [ ]:
# @title Install dependencies
!pip install -q ultralytics roboflow loguru typer python-dotenv pyyaml matplotlib opencv-python-headless albumentations

In [ ]:
# @title Mount Drive or clone repo
import os
from pathlib import Path
import sys

!git clone https://github.com/Hndra04/AlGear
PROJECT_DIR = Path("/content/AlGear")

os.chdir(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project root: {PROJECT_DIR}")

In [ ]:
# @title Set Roboflow API key
import os
os.environ["ROBOFLOW_API_KEY"] = ""

from algear.config import ROBOFLOW_DIR, ROBOFLOW_API_KEY
print(f"API key set: {bool(ROBOFLOW_API_KEY)}")
print(f"Dataset at: {ROBOFLOW_DIR}")

In [ ]:
# @title Download dataset (if not already present)
if not ROBOFLOW_DIR.exists():
    from algear.dataset import download_roboflow
    download_roboflow(output_dir=ROBOFLOW_DIR)
else:
    print(f"Dataset already exists at {ROBOFLOW_DIR}")

In [ ]:
# @title Check GPU availability
import torch

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Resplit Dataset

If the resplit dataset doesn't exist yet, run notebook 3.0 first.

In [ ]:
# @title Resplit dataset (skip if already done)
from algear.dataset import resplit

RESPLIT_DIR = PROJECT_DIR / "data" / "processed" / "construction-site-safety-resplit"

if not (RESPLIT_DIR / "data.yaml").exists():
    print("Running resplit...")
    resplit(
        src_dir=ROBOFLOW_DIR,
        output_dir=RESPLIT_DIR,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
        seed=42,
    )
else:
    print(f"Resplit dataset already exists at {RESPLIT_DIR}")

In [ ]:
# @title Set paths
MODEL_PATH = PROJECT_DIR / "models" / "resplit-oversample-conservative" / "weights" / "best.pt"
TEST_IMAGE_DIR = RESPLIT_DIR / "test" / "images"
TEST_LABEL_DIR = RESPLIT_DIR / "test" / "labels"

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert TEST_IMAGE_DIR.exists(), f"Test images not found: {TEST_IMAGE_DIR}"
assert TEST_LABEL_DIR.exists(), f"Test labels not found: {TEST_LABEL_DIR}"

print(f"Model: {MODEL_PATH}")
print(f"Test images: {TEST_IMAGE_DIR}")
print(f"Test labels: {TEST_LABEL_DIR}")

## 2. Run Inference on Test Set

In [ ]:
# @title Run inference on all test images
from algear.modeling.predict import run_inference

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_images = sorted(f for f in TEST_IMAGE_DIR.iterdir() if f.suffix.lower() in IMAGE_EXTS)
print(f"Test images: {len(test_images)}")

pred_results = run_inference(
    model_path=MODEL_PATH,
    image_dir=TEST_IMAGE_DIR,
    conf=0.25,
    device=device,
)

print(f"Results for {len(pred_results)} images")

## 3. People Counting MAE

In [ ]:
# @title Count GT persons per image
PERSON_CLASS = 3

def count_persons_in_label(label_path: Path) -> int:
    """Count person instances in a YOLO label file."""
    if not label_path.exists():
        return 0
    count = 0
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5 and int(parts[0]) == PERSON_CLASS:
                count += 1
    return count

gt_counts = []
for img_path in test_images:
    label_path = TEST_LABEL_DIR / f"{img_path.stem}.txt"
    gt_counts.append(count_persons_in_label(label_path))

print(f"GT person counts — total: {sum(gt_counts)}, mean: {np.mean(gt_counts):.1f}, max: {max(gt_counts)}")

In [ ]:
# @title Count predicted persons per image
pred_counts = []
for r in pred_results:
    if len(r["detections"]) == 0:
        pred_counts.append(0)
    else:
        pred_counts.append(int((r["detections"][:, 0] == PERSON_CLASS).sum()))

print(f"Pred person counts — total: {sum(pred_counts)}, mean: {np.mean(pred_counts):.1f}, max: {max(pred_counts)}")

In [ ]:
# @title Compute People Counting MAE
from algear.metrics import people_counting_mae

mae_result = people_counting_mae(gt_counts, pred_counts)

print(f"\n=== People Counting Results ===")
print(f"MAE:              {mae_result['mae']:.2f}  (target: \u2264 2)")
print(f"Mean GT count:    {mae_result['mean_gt']:.1f}")
print(f"Mean pred count:  {mae_result['mean_pred']:.1f}")
print(f"Max error:        {mae_result['max_error']}")
print(f"Status:           {'\u2705 PASS' if mae_result['mae'] <= 2 else '\u274c FAIL'}")

In [ ]:
# @title Visualise People Counting results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: GT vs Predicted
axes[0].scatter(gt_counts, pred_counts, alpha=0.5, s=20, c="steelblue")
max_val = max(max(gt_counts), max(pred_counts)) + 1
axes[0].plot([0, max_val], [0, max_val], "r--", label="Perfect")
axes[0].set_xlabel("Ground Truth Person Count")
axes[0].set_ylabel("Predicted Person Count")
axes[0].set_title(f"People Counting (MAE={mae_result['mae']:.2f})")
axes[0].legend()
axes[0].set_xlim(-0.5, max_val)
axes[0].set_ylim(-0.5, max_val)
axes[0].set_aspect("equal")

# Error distribution
errors = [r["error"] for r in mae_result["per_image"]]
axes[1].hist(errors, bins=range(0, max(errors) + 2), color="steelblue", edgecolor="white")
axes[1].axvline(mae_result["mae"], color="red", linestyle="--", label=f"MAE={mae_result['mae']:.2f}")
axes[1].set_xlabel("Absolute Error (persons)")
axes[1].set_ylabel("Number of Images")
axes[1].set_title("Error Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. PPE Compliance Accuracy

In [ ]:
# @title Compute PPE Compliance Accuracy
from algear.metrics import ppe_compliance_accuracy

compliance_result = ppe_compliance_accuracy(
    gt_labels_dir=TEST_LABEL_DIR,
    pred_results=pred_results,
    img_dir=TEST_IMAGE_DIR,
)

print(f"\n=== PPE Compliance Results ===")
print(f"Accuracy:         {compliance_result['accuracy']:.1%}  (target: \u2265 85%)")
print(f"Total persons:    {compliance_result['total_persons']}")
print(f"Correct:          {compliance_result['correct']}")
print(f"Status:           {'\u2705 PASS' if compliance_result['accuracy'] >= 0.85 else '\u274c FAIL'}")

In [ ]:
# @title Per-image compliance detail
per_img = compliance_result["per_image"]
images_with_errors = [p for p in per_img if p["gt_persons"] > 0 and p["correct"] < p["total"]]
print(f"Images with compliance errors: {len(images_with_errors)} / {len(per_img)}")

# Worst cases
worst = sorted(per_img, key=lambda x: x["total"] - x["correct"], reverse=True)[:10]
print("\nWorst 10 images (most misclassifications):")
print(f"{'Image':<35s} {'GT':>4s} {'Pred':>5s} {'Correct':>7s} {'Total':>5s}")
for w in worst:
    print(f"{w['image']:<35s} {w['gt_persons']:>4d} {w['pred_persons']:>5d} {w['correct']:>7d} {w['total']:>5d}")

In [ ]:
# @title Visualise PPE Compliance results
n_correct = compliance_result["correct"]
n_total = compliance_result["total_persons"]
n_wrong = n_total - n_correct

fig, ax = plt.subplots(figsize=(5, 5))
ax.bar(["Correct", "Misclassified"], [n_correct, n_wrong], color=["#2ecc71", "#e74c3c"])
ax.set_ylabel("Person count")
ax.set_title(f"PPE Compliance \u2014 {compliance_result['accuracy']:.1%} accuracy")
for i, v in enumerate([n_correct, n_wrong]):
    ax.text(i, v + 5, str(v), ha="center", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Combined Metrics Summary

In [ ]:
# @title Metrics summary table
print("=" * 70)
print("  PIPELINE METRICS SUMMARY  (Test Set)")
print("=" * 70)
print()
print(f"  {'Metric':<40s} {'Target':>10s} {'Result':>10s} {'Status':>6s}")
print(f"  {'-'*40} {'-'*10} {'-'*10} {'-'*6}")

mae_ok = mae_result["mae"] <= 2
acc_ok = compliance_result["accuracy"] >= 0.85

print(f"  {'People Counting MAE':<40s} {'\u2264 2':>10s} {mae_result['mae']:>10.2f} {'\u2705' if mae_ok else '\u274c':>6s}")
print(f"  {'PPE Compliance Accuracy':<40s} {'\u2265 85%':>10s} {compliance_result['accuracy']:>9.1%} {'\u2705' if acc_ok else '\u274c':>6s}")
print()

# Detection-level metrics from notebook 3.1
print("  Detection-level metrics (from notebook 3.1):")
print(f"  {'mAP@50 (overall)':<40s} {'\u2265 80%':>10s} {'81.0%':>10s} {'\u2705':>6s}")
print(f"  {'FNR no-helmet':<40s} {'\u2264 10%':>10s} {'36.2%':>10s} {'\u274c':>6s}")
print(f"  {'FNR no-vest':<40s} {'\u2264 10%':>10s} {'18.1%':>10s} {'\u274c':>6s}")
print()
print("=" * 70)

In [ ]:
# @title Download model (optional)
from google.colab import files
files.download(str(MODEL_PATH))